<a href="https://colab.research.google.com/github/GIRIAYUSH/playing-with-anns/blob/main/notebooks/M5_Gradient_Descent_and_Optimizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
print("All imports ready ")

## M5 · Gradient Descent & Optimizers

> *"Learning is just rolling a ball downhill on a surface made of your mistakes."*

Before diving in, here's the big picture:

- **Loss surface**: A landscape where each point (w₁, w₂) has a height = loss value
- **Gradient**: The slope at your current position — tells you which direction is *uphill*
- **Optimizer**: The strategy for taking steps *downhill* toward the minimum

We'll go from hand-coded vanilla GD all the way to Adam, and build intuition for *why* Adam works.

### 3D Loss Surface Visualization

In [ ]:
# 3D Loss Surface
fig = plt.figure(figsize=(14, 6))

# A bowl-shaped loss surface: L(w1, w2) = w1^2 + 2*w2^2
w1 = np.linspace(-3, 3, 200)
w2 = np.linspace(-3, 3, 200)
W1, W2 = np.meshgrid(w1, w2)
L = W1**2 + 2 * W2**2  # simple convex surface

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W1, W2, L, cmap='plasma', alpha=0.85, linewidth=0)
ax1.set_xlabel('w₁'); ax1.set_ylabel('w₂'); ax1.set_zlabel('Loss')
ax1.set_title('3D Loss Surface\nL = w₁² + 2w₂²')
fig.colorbar(surf, ax=ax1, shrink=0.4)

# 2D contour
ax2 = fig.add_subplot(122)
cp = ax2.contourf(W1, W2, L, levels=30, cmap='plasma')
ax2.contour(W1, W2, L, levels=30, colors='white', linewidths=0.3, alpha=0.4)
ax2.set_xlabel('w₁'); ax2.set_ylabel('w₂')
ax2.set_title('2D Contour Map\n(top-down view)')
fig.colorbar(cp, ax=ax2)

plt.tight_layout()
plt.show()

## The Loss Surface

The loss surface is the terrain we're trying to navigate. For a model with two weights (w₁, w₂):

- Each (w₁, w₂) pair = a specific model configuration
- The **height** at that point = how bad the model performs (loss)
- The **minimum** (bottom of the bowl) = best weights

**Contour lines** = rings of equal loss. When contours are close together → steep gradient. When far apart → gentle slope.

Our surface here is `L = w₁² + 2w₂²` — an elliptical bowl. The global minimum sits at (0, 0). Notice how w₂ has a steeper curvature (coefficient 2) — this asymmetry will make some optimizers struggle more than others.

### Understanding Local Minimas
Real neural network loss surfaces are **non-convex** — they have:

| Feature | Description |
|---|---|
| **Local minima** | Valleys that aren't the deepest valley |
| **Saddle points** | Flat regions where gradient ≈ 0 but not a minimum |
| **Plateaus** | Vast flat areas where learning stalls |
| **Ravines** | Narrow, curved valleys (hard for vanilla GD) |

A simple optimizer starting at the wrong point can get **stuck** in a local minimum and never reach the global one. This is why momentum, adaptive learning rates, and stochasticity matter — they help escape these traps.

> Interestingly, for very deep networks, most local minima have *similar loss* to the global minimum — saddle points are the bigger problem in practice.

In [ ]:
# Loss surface WITH local minima
w1 = np.linspace(-4, 4, 300)
w2 = np.linspace(-4, 4, 300)
W1, W2 = np.meshgrid(w1, w2)

# Non-convex surface: multiple minima
L_nonconvex = (np.sin(W1) * np.cos(W2) +
               0.1 * W1**2 + 0.1 * W2**2 +
               np.sin(2*W1) * 0.3)

fig = plt.figure(figsize=(15, 6))

ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W1, W2, L_nonconvex, cmap='coolwarm', alpha=0.85)
ax1.set_xlabel('w₁'); ax1.set_ylabel('w₂'); ax1.set_zlabel('Loss')
ax1.set_title('Non-Convex Surface\n(Local Minima Problem)')
ax1.view_init(elev=35, azim=45)

ax2 = fig.add_subplot(122)
cp = ax2.contourf(W1, W2, L_nonconvex, levels=40, cmap='coolwarm')
ax2.contour(W1, W2, L_nonconvex, levels=40, colors='white', linewidths=0.2, alpha=0.3)

# Mark some local minima (approximate)
local_mins = [(-1.6, 0.0), (1.6, 0.0), (0.0, -1.6), (0.0, 1.6)]
for lm in local_mins:
    ax2.plot(*lm, 'y*', markersize=12, label='Local min')
ax2.plot(0, 0, 'g*', markersize=15, label='Global min (approx)')
ax2.set_xlabel('w₁'); ax2.set_ylabel('w₂')
ax2.set_title('Contour + Local Minima')
ax2.legend(loc='upper right', fontsize=8)
fig.colorbar(cp, ax=ax2)

plt.tight_layout()
plt.show()

### Vanilla Gradient Descent


The update rule is beautifully simple:

$$w \leftarrow w - \eta \cdot \nabla_w L$$

Where:
- $w$ = current weight
- $\eta$ (eta) = **learning rate** — how big a step to take
- $\nabla_w L$ = gradient of loss w.r.t. weight — which direction is uphill

**Every optimizer you'll ever use is a variation of this one line.**

The gradient tells us the direction of steepest ascent, so we subtract it to go downhill. After enough steps, we converge to the minimum.

**Limitations of vanilla GD:**
- Uses the *full dataset* to compute each gradient → slow for large datasets
- Same learning rate for all weights → struggles with ill-conditioned surfaces (ravines)
- No memory of past gradients → can oscillate

In [ ]:
# Manual Gradient Descent from scratch

def loss_fn(w1, w2):
    """L = w1^2 + 2*w2^2  →  minimum at (0, 0)"""
    return w1**2 + 2 * w2**2

def grad_fn(w1, w2):
    """∂L/∂w1 = 2w1,  ∂L/∂w2 = 4w2"""
    return 2 * w1, 4 * w2

# Hyperparameters
lr = 0.15
num_steps = 40
w1, w2 = 2.5, 2.5  # starting point

path_vanilla = [(w1, w2, loss_fn(w1, w2))]

for step in range(num_steps):
    g1, g2 = grad_fn(w1, w2)
    w1 = w1 - lr * g1   # ← THE CORE UPDATE RULE
    w2 = w2 - lr * g2
    path_vanilla.append((w1, w2, loss_fn(w1, w2)))

path_vanilla = np.array(path_vanilla)

# Plot the path on contour
W1g, W2g = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
Lg = W1g**2 + 2 * W2g**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour + path
ax = axes[0]
ax.contourf(W1g, W2g, Lg, levels=25, cmap='plasma', alpha=0.8)
ax.contour(W1g, W2g, Lg, levels=25, colors='white', linewidths=0.3, alpha=0.4)
ax.plot(path_vanilla[:, 0], path_vanilla[:, 1], 'o-', color='cyan',
        linewidth=2, markersize=4, label='GD path')
ax.plot(*path_vanilla[0, :2], 'go', markersize=10, label='Start')
ax.plot(*path_vanilla[-1, :2], 'r*', markersize=12, label='End')
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title(f'Vanilla GD Path (lr={lr})')
ax.legend()

# Loss curve
ax = axes[1]
ax.plot(path_vanilla[:, 2], 'b-o', markersize=3, linewidth=2)
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('Loss over Steps')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final weights: w1={w1:.6f}, w2={w2:.6f}")
print(f"Final loss:    {loss_fn(w1, w2):.8f}")

### PyTorch Built-in Gradient Descent

In [ ]:
# Gradient Descent using PyTorch autograd

torch.manual_seed(42)

# Our "model": just two learnable parameters
w = torch.tensor([2.5, 2.5], requires_grad=True, dtype=torch.float32)

optimizer = torch.optim.SGD([w], lr=0.15)

path_torch = [w.detach().numpy().copy()]
losses_torch = []

for step in range(40):
    optimizer.zero_grad()           # 1. Clear old gradients
    loss = w[0]**2 + 2 * w[1]**2   # 2. Forward pass: compute loss
    loss.backward()                  # 3. Backward pass: compute gradients
    optimizer.step()                 # 4. Update weights: w = w - lr * w.grad

    path_torch.append(w.detach().numpy().copy())
    losses_torch.append(loss.item())

path_torch = np.array(path_torch)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.contourf(W1g, W2g, Lg, levels=25, cmap='viridis', alpha=0.8)
ax.plot(path_torch[:, 0], path_torch[:, 1], 'o-', color='orange',
        linewidth=2, markersize=4)
ax.plot(*path_torch[0], 'go', markersize=10, label='Start')
ax.plot(*path_torch[-1], 'r*', markersize=12, label='End')
ax.set_title('PyTorch SGD Path'); ax.legend()
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')

ax = axes[1]
ax.plot(losses_torch, 'orange', linewidth=2)
ax.set_xlabel('Step'); ax.set_ylabel('Loss (log scale)')
ax.set_title('Loss Curve (PyTorch SGD)')
ax.set_yscale('log'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Confirm gradients were computed correctly
w_check = torch.tensor([2.5, 2.5], requires_grad=True, dtype=torch.float32)
loss_check = w_check[0]**2 + 2*w_check[1]**2
loss_check.backward()
print(f"Manual gradient at (2.5, 2.5): dL/dw1={2*2.5}, dL/dw2={4*2.5}")
print(f"PyTorch gradient at (2.5, 2.5): {w_check.grad.tolist()}")

## PyTorch's Autograd — The 4-Step Dance

Every training loop in PyTorch follows this exact pattern:
- optimizer.zero_grad()   # 1. Wipe old gradients (they accumulate by default!)
- loss = forward(x)       # 2. Compute loss
- loss.backward()         # 3. Backprop: fill .grad for every parameter
- optimizer.step()        # 4. Apply update rule

**Why `zero_grad()`?** PyTorch *adds* gradients to `.grad` each backward pass. If you forget this, gradients pile up across steps → wrong updates.

**Why separate `.backward()` and `.step()`?** It gives you control: inspect gradients, clip them, log them, modify them — all before the optimizer touches the weights.

Notice that PyTorch's result matches our manual computation exactly. `autograd` is just automatic differentiation — not magic.

### Learning Rate Experiments: Playing with Different Learning Rates to visualize Gradient Descent



| LR too small | LR just right | LR too large |
|---|---|---|
| Tiny steps, slow convergence | Smooth descent to minimum | Overshoots, oscillates or diverges |
| May never reach minimum | Finds minimum reliably | Loss *increases* over time |
| Wastes compute |  | Unstable training |

**Why does too-high LR diverge?**  
If the step is larger than the curvature of the loss surface allows, you overshoot the minimum and land on the *other side* — which is even steeper. Next step overshoots further. The ball bounces out of the bowl entirely.

**Mathematical condition for convergence:**
$$\eta < \frac{2}{\lambda_{max}}$$
where $\lambda_{max}$ is the largest eigenvalue of the Hessian (curvature matrix). In practice, we tune LR empirically.

> Rule of thumb starting points: `1e-3` for Adam, `1e-2` to `1e-1` for SGD.

## Vanishing Gradients — Why Deep Networks Are Hard

During backpropagation, gradients are multiplied layer by layer via the chain rule:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial a_n} \cdot \frac{\partial a_n}{\partial a_{n-1}} \cdots \frac{\partial a_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial w_1}$$

**The problem:** Sigmoid's derivative maxes out at 0.25:
$$\sigma'(x) = \sigma(x)(1 - \sigma(x)) \leq 0.25$$

Multiply 10 layers: $0.25^{10} \approx 10^{-6}$ — essentially zero! Early layers receive nearly no gradient signal → they stop learning.

**SGD makes this worse** because it applies the same learning rate to all parameters. A layer with gradient `1e-8` needs a *much* higher effective LR than a layer with gradient `1.0` — but SGD can't distinguish.

**Solutions:**
- ReLU activation: derivative = 1 for positive inputs (no shrinkage)
- Batch normalization: re-centers activations each layer
- Residual connections (ResNets): shortcuts that bypass layers
- Better initialization (Xavier, He init)
- **Adaptive optimizers (Adam)**: per-parameter LR rescues tiny gradients

In [ ]:
# Vanishing Gradients through a Deep Network

torch.manual_seed(42)

def build_deep_net(depth=10, activation=nn.Sigmoid()):
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(10, 10), activation]
    layers.append(nn.Linear(10, 1))
    return nn.Sequential(*layers)

def get_gradient_norms(net, x, y):
    criterion = nn.MSELoss()
    out = net(x)
    loss = criterion(out, y)
    loss.backward()
    norms = []
    for name, param in net.named_parameters():
        if 'weight' in name and param.grad is not None:
            norms.append((name, param.grad.norm().item()))
    return norms

x = torch.randn(32, 10)
y = torch.randn(32, 1)

# Sigmoid (vanishing) vs ReLU (healthier)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (act_name, act) in zip(axes, [('Sigmoid (Vanishing!)', nn.Sigmoid()),
                                        ('ReLU (Healthier)', nn.ReLU())]):
    net = build_deep_net(depth=10, activation=act)
    # Init weights small to exaggerate effect
    for p in net.parameters():
        nn.init.normal_(p, 0, 0.5)

    norms = get_gradient_norms(net, x, y)

    layer_names = [n.replace('.weight', '').replace('module.', '') for n, _ in norms]
    grad_values = [v for _, v in norms]

    bars = ax.bar(range(len(grad_values)), grad_values,
                  color=plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(grad_values))))
    ax.set_xticks(range(len(grad_values)))
    ax.set_xticklabels([f'L{i+1}' for i in range(len(grad_values))], fontsize=8)
    ax.set_xlabel('Layer (L1=earliest, deeper=later)')
    ax.set_ylabel('Gradient Norm')
    ax.set_title(f'{act_name}\nGradient norms per layer')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')

    # Annotate
    for i, v in enumerate(grad_values):
        ax.text(i, v * 1.3, f'{v:.1e}', ha='center', fontsize=6, rotation=45)

plt.suptitle('Vanishing Gradients: Sigmoid vs ReLU', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Optimizer Mathematics — All Four, Side by Side

---

### 1. Vanilla SGD
$$w_t = w_{t-1} - \eta \cdot g_t$$

- $g_t = \nabla_w L$ (gradient at step $t$)  
- Simple, interpretable, no memory  
- **Problem:** Same LR for all parameters, sensitive to noise

---

### 2. SGD + Momentum
$$v_t = \beta v_{t-1} + g_t \qquad w_t = w_{t-1} - \eta \cdot v_t$$

- $v_t$ = velocity (exponential moving average of gradients)  
- $\beta$ = momentum coefficient (typically 0.9)  
- **Intuition:** A ball rolling downhill *accumulates speed*. It doesn't stop at every tiny bump; it rolls through noise and oscillations  
- Accelerates along consistent gradient directions, dampens oscillations

---

### 3. RMSProp
$$s_t = \rho \cdot s_{t-1} + (1-\rho) \cdot g_t^2 \qquad w_t = w_{t-1} - \frac{\eta}{\sqrt{s_t + \epsilon}} \cdot g_t$$

- $s_t$ = exponential moving average of *squared* gradients  
- **Key idea:** Divide LR by root of recent gradient magnitude → **adaptive per-parameter learning rate**  
- Parameters with large gradients get smaller effective LR; rare/small-gradient params get larger LR  
- Excellent for non-stationary problems (RNNs)

---

### 4. Adam (Adaptive Moment Estimation)
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(RMSProp)}$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{(bias correction)}$$
$$w_t = w_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

- **Adam = Momentum + RMSProp + bias correction**  
- Defaults: $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$  
- Bias correction: early steps have too few samples for averages → scale them up  
- Works well out of the box for most problems

In [ ]:
# Compare SGD / Momentum / RMSProp / Adam on elliptical bowl

def run_optimizer(opt_name, steps=80):
    w = torch.tensor([2.5, 2.5], requires_grad=True, dtype=torch.float32)

    opts = {
        'SGD':      torch.optim.SGD([w], lr=0.08),
        'Momentum': torch.optim.SGD([w], lr=0.08, momentum=0.9),
        'RMSProp':  torch.optim.RMSprop([w], lr=0.08),
        'Adam':     torch.optim.Adam([w], lr=0.3),
    }
    opt = opts[opt_name]

    path = [w.detach().numpy().copy()]
    losses = []

    for _ in range(steps):
        opt.zero_grad()
        loss = w[0]**2 + 2 * w[1]**2
        loss.backward()
        opt.step()
        path.append(w.detach().numpy().copy())
        losses.append(loss.item())

    return np.array(path), losses

optimizer_names = ['SGD', 'Momentum', 'RMSProp', 'Adam']
colors_map = {'SGD': 'cyan', 'Momentum': 'orange', 'RMSProp': 'magenta', 'Adam': 'limegreen'}

results = {name: run_optimizer(name) for name in optimizer_names}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i, name in enumerate(optimizer_names):
    path, losses = results[name]

    # Contour path
    ax = axes[0, i]
    ax.contourf(W1g, W2g, Lg, levels=25, cmap='plasma', alpha=0.7)
    ax.contour(W1g, W2g, Lg, levels=25, colors='white', linewidths=0.2, alpha=0.3)
    ax.plot(path[:, 0], path[:, 1], 'o-', color=colors_map[name],
            linewidth=1.8, markersize=2.5)
    ax.plot(*path[0], 'go', markersize=9, label='Start')
    ax.plot(*path[-1], 'r*', markersize=10, label='End')
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
    ax.legend(fontsize=7)

    # Loss curve
    ax = axes[1, i]
    ax.plot(losses, color=colors_map[name], linewidth=2)
    ax.set_yscale('log')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title(f'{name} Loss')
    ax.grid(True, alpha=0.3)
    ax.text(0.6, 0.85, f'Final: {losses[-1]:.2e}', transform=ax.transAxes, fontsize=9)

plt.suptitle('Optimizer Comparison on Elliptical Bowl', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## The Race — What to Notice

Watching all four optimizers simultaneously reveals their personalities:

| Optimizer | Path Shape | Speed | Notes |
|---|---|---|---|
| **SGD** | Straight but slow | Slowest | Steady, no flair |
| **Momentum** | Curves, then overshoots | Fast | "Ball rolling" — builds speed, may oscillate |
| **RMSProp** | Adapts per-axis | Medium | Handles asymmetric bowl better |
| **Adam** | Confident, direct | Fastest to converge | Combines both advantages |

The **elliptical bowl** is a perfect test because the w₂ axis is steeper (coefficient 2). SGD treats both axes identically → takes suboptimal path. Adam's per-axis adaptation means it can take larger steps along w₁ and smaller ones along w₂.

In [ ]:
# Animation: all 4 optimizers racing to minimum

fig, ax = plt.subplots(figsize=(8, 7))
ax.contourf(W1g, W2g, Lg, levels=25, cmap='plasma', alpha=0.8)
ax.contour(W1g, W2g, Lg, levels=25, colors='white', linewidths=0.3, alpha=0.3)
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Optimizer Race ')

lines = {}
dots  = {}
for name in optimizer_names:
    line, = ax.plot([], [], '-', color=colors_map[name], linewidth=2, label=name)
    dot,  = ax.plot([], [], 'o', color=colors_map[name], markersize=8)
    lines[name] = line
    dots[name]  = dot

ax.legend(loc='upper right')
ax.plot(0, 0, 'w*', markersize=15, zorder=5, label='Target')

def init():
    for name in optimizer_names:
        lines[name].set_data([], [])
        dots[name].set_data([], [])
    return list(lines.values()) + list(dots.values())

def update(frame):
    for name in optimizer_names:
        path = results[name][0]
        end = min(frame + 1, len(path))
        lines[name].set_data(path[:end, 0], path[:end, 1])
        dots[name].set_data([path[end-1, 0]], [path[end-1, 1]])
    return list(lines.values()) + list(dots.values())

ani = animation.FuncAnimation(fig, update, frames=80, init_func=init,
                               interval=80, blit=True)
plt.tight_layout()
plt.show()
ani.save('optimizer_race.gif', writer='pillow', fps=15)

## Learning Rate Schedulers — Warming Down

A fixed LR is a compromise: big enough to learn quickly, small enough not to overshoot. Schedulers let you **have both** — start high, end low.

### StepLR
$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / \text{step\_size} \rfloor}$$
- Drops LR by factor $\gamma$ every `step_size` epochs  
- Simple, widely used in vision models (ResNet training)  
- Creates a **staircase** schedule

### CosineAnnealingLR
$$\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})\left(1 + \cos\frac{\pi t}{T}\right)$$
- Smooth cosine curve from `lr_max` down to `lr_min`  
- More natural, no sudden drops  
- Popular in modern training (BERT, ViT, diffusion models)

### Why schedule at all?
- **Early training**: Large LR → fast coarse movement across loss landscape  
- **Late training**: Small LR → fine-grained convergence without oscillating around minimum  
- Without scheduling, you're stuck picking one LR for both jobs

> A common modern recipe: **Adam + Cosine schedule + warmup**. Start tiny (warmup), rise to peak LR, then cosine anneal down.

In [ ]:
torch.manual_seed(0)

def train_with_scheduler(sched_name, steps=100):
    w = torch.tensor([2.5, 2.5], requires_grad=True, dtype=torch.float32)
    opt = torch.optim.SGD([w], lr=0.3)

    if sched_name == 'StepLR':
        scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=25, gamma=0.5)
    elif sched_name == 'CosineAnnealing':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    else:
        scheduler = None

    losses, lrs = [], []

    for step in range(steps):
        opt.zero_grad()
        loss = w[0]**2 + 2 * w[1]**2
        loss.backward()
        opt.step()
        if scheduler:
            scheduler.step()
        losses.append(loss.item())
        lrs.append(opt.param_groups[0]['lr'])

    return losses, lrs

sched_names = ['None (constant)', 'StepLR', 'CosineAnnealing']
sched_keys  = [None, 'StepLR', 'CosineAnnealing']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, (name, key) in enumerate(zip(sched_names, sched_keys)):
    if key is None:
        w = torch.tensor([2.5, 2.5], requires_grad=True, dtype=torch.float32)
        opt = torch.optim.SGD([w], lr=0.3)
        losses, lrs = [], []
        for _ in range(100):
            opt.zero_grad()
            loss = w[0]**2 + 2*w[1]**2
            loss.backward(); opt.step()
            losses.append(loss.item()); lrs.append(0.3)
    else:
        losses, lrs = train_with_scheduler(key)

    axes[0, i].plot(losses, 'b', linewidth=2)
    axes[0, i].set_title(f'{name}\nLoss'); axes[0, i].set_yscale('log')
    axes[0, i].set_xlabel('Step'); axes[0, i].grid(True, alpha=0.3)

    axes[1, i].plot(lrs, 'r', linewidth=2)
    axes[1, i].set_title(f'Learning Rate Schedule')
    axes[1, i].set_xlabel('Step'); axes[1, i].set_ylabel('LR')
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('LR Schedulers: Loss + Schedule Curves', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()